### Exercise — End to End Preprocessing Pipeline

Build a complete preprocessing pipeline on a real CSV dataset
in the following sequence:

1. EDA first — before touching anything run:
   - df.info()
   - df.shape
   - df.isnull().sum()/ * 100
   - df.describe()
   - df.duplicated().sum()
   - df.sample(10)

2. Based on what EDA reveals, make decisions and write them as comments:
   - Which columns to drop and why
   - Which missing values to fill and with what
   - Which columns need type conversion
   - Which text columns need standardising

3. Execute the cleaning in this order:
   - Remove duplicates
   - Drop columns that are too sparse or irrelevant
   - Fill missing numeric values
   - Fill missing categorical values
   - Standardise text columns
   - Reset index

4. Run df.info() and df.shape again at the end
   Compare with step 1 

## Answer

## Titanic Dataset — Preprocessing Pipeline

The Titanic dataset contains passenger information from the 1912 disaster.
The goal is to predict whether a passenger survived (1) or did not survive (0)
based on features such as age, sex, fare, and passenger class.

Before feeding this data to a model, it must be cleaned. Raw data contains
missing values, irrelevant columns, and inconsistencies that would reduce
model accuracy or cause errors during training.

Conducting EDA

In [31]:
import pandas as pd

df = pd.read_csv('Titanic-Dataset.csv')

df.info()
print('-' * 40)
print(df.shape)
print('-' * 40)
print(df.isnull().sum()/len(df) * 100)
print('-' * 40)
print(df.describe())
print('-' * 40)
print(df.duplicated().sum())
print('-' * 40)
print(df.sample(10))


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
----------------------------------------
(891, 12)
----------------------------------------
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0

### Name, Ticket, PassengerId, Cabin: not useful for ML — drop columns

**PassengerId** — a sequential identifier assigned arbitrarily.
It has no relationship to survival and would cause the model
to memorise row numbers rather than learn real patterns.

**Ticket** — ticket numbers have no consistent format and carry
no meaningful signal that predicts survival.

**Name** — as a raw column, names do not predict survival.
The titles embedded in names (Mr., Mrs., Master.) do carry
signal, but extracting them is a feature engineering step
beyond the scope of this pipeline.

**Cabin** — 77% of values are missing. A column this sparse
cannot contribute reliable information to a model and is
dropped entirely.

In [32]:
df = df.drop(columns=['PassengerId', 'Ticket', 'Name', 'Cabin'])
df.shape

(891, 8)

Result: No. of columns down by 4

### Age: 19.9% missing — fill with median


In [33]:
df['Age'] = df['Age'].fillna(df['Age'].median())

### Embarked: 2 rows missing — fill with most frequent port 


In [34]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,28.0,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


Ratio of female survivors to total no. female

In [ ]:
female_survivors = ((df['Survived'] == 1) & (df['Sex'] == 'female')).sum()
ratio1 = female_survivors / (df['Sex'] == 'female').sum()
print(ratio1)

0.7420382165605095


Ratio of female survivors to total no. female

In [46]:
male_survivors = ((df['Survived'] == 1) & (df['Sex'] == 'male')).sum()
ratio2 = male_survivors / (df['Sex'] == 'male').sum()
print(ratio2)

0.18890814558058924
